# J1S4 — Nettoyage & Feature Engineering
## BankRisk Intelligence Platform · Contexte bancaire ivoirien

**Objectif :** Produire `credit_risk_clean.parquet` — dataset ML-ready pour tout le Jour 2.  
**Entrée :** `credit_features_j1.parquet` (32 581 lignes × 15 colonnes, produit par J1S2).  
**Sortie :** `credit_risk_clean.parquet` (~22 colonnes, 0 NaN, toutes variables numériques).  

---

### Chaîne Parquet BankRisk
```
credit_risk_dataset.csv          (12 cols originales)
  → drop(loan_grade) ✗           (variable pré-octroi externe LendingClub)
      → credit_features_j1.parquet   ← ENTRÉE J1S4 (15 cols)
          → credit_risk_clean.parquet ← LIVRABLE J1S4 (~22 cols, ML-ready)
              → credit_risk_kmeans.parquet  (J2S1, K-Means)
                  → sorties RF/LR           (J2S2, AUC=0,929)
```

### Pourquoi loan_grade est absente ?
> **Décision documentée en J1S2** — `loan_grade` est attribuée par LendingClub *après* évaluation du risque.  
> En banque ivoirienne, aucun grade externe n'existe : la banque doit construire son scoring depuis les variables sources.  
> L'inclure constituerait un **data leakage conceptuel** qui fausse l'évaluation réelle du modèle.

### Taux de défaut de référence : **21,8 %** (benchmark de surrisque BCEAO)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 0 — Setup · ROOT detection + pip install + imports (cellule unique)
# RÈGLE ABSOLUE : ne jamais diviser cette cellule — NameError garanti
# ═══════════════════════════════════════════════════════════════════
import sys, os
from pathlib import Path

# Détection environnement : Google Colab ou VS Code local
try:
    from google.colab import drive
    IN_COLAB = True
    # Monter Google Drive si Colab
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/bankrisk')
except ImportError:
    IN_COLAB = False
    # VS Code local : remonter jusqu'à la racine du projet
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'data').exists() or (ROOT / 'requirements.txt').exists():
            break
        ROOT = ROOT.parent

print(f"Environnement : {'Google Colab' if IN_COLAB else 'VS Code local'}")
print(f"ROOT : {ROOT}")

# Installation des dépendances depuis requirements.txt
req_file = ROOT / 'requirements.txt'
if req_file.exists():
    os.system(f'{sys.executable} -m pip install -r {req_file} -q')
else:
    os.system(f'{sys.executable} -m pip install pandas numpy scikit-learn plotly pyarrow -q')

# ── Imports ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import subprocess

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"plotly  : {plotly.__version__}")  # plotly.__version__, pas px.__version__
print(f"sklearn : {__import__('sklearn').__version__}")

# Chemins Parquet (toujours ROOT / 'data' / 'processed' / 'fichier.parquet')
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

PARQUET_IN  = DATA_PROCESSED / 'credit_features_j1.parquet'
PARQUET_OUT = DATA_PROCESSED / 'credit_risk_clean.parquet'

# Palette Ocean Executive BankRisk
NAVY   = '#021B2E'
DEEP   = '#065A82'
TEAL   = '#1C7293'
MINT   = '#02C39A'
ORANGE = '#FFA07A'

print("\n✓ Setup complet — J1S4 prêt")

---
## Bloc 1 — Chargement et assertions

> **Contexte métier :** Avant toute transformation, on vérifie que le parquet entrant est conforme à ce qu'on attend :  
> 32 581 observations, 15 colonnes, sans `loan_grade`, avec `debt_service_rate` déjà calculé en J1S2.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 1 — Chargement et assertions qualité
# ═══════════════════════════════════════════════════════════════════

# Si le parquet J1S2 n'existe pas encore (exécution standalone),
# on peut le reconstruire depuis le CSV source
if not PARQUET_IN.exists():
    print("⚠ credit_features_j1.parquet introuvable — reconstruction depuis le CSV...")
    csv_path = DATA_RAW / 'credit_risk_dataset.csv'
    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV source introuvable : {csv_path}\n"
            "Placez credit_risk_dataset.csv dans data/raw/ ou exécutez J1S2 d'abord."
        )
    df_raw = pd.read_csv(csv_path)
    # Retrait loan_grade (décision J1S2 — variable pré-octroi LendingClub)
    df_raw = df_raw.drop(columns=['loan_grade'], errors='ignore')
    # Features J1S2 minimales pour la suite
    df_raw['debt_service_rate']   = df_raw['loan_int_rate'] * df_raw['loan_percent_income']
    df_raw['monthly_payment_proxy'] = df_raw['loan_amnt'] / (df_raw['person_income'] / 12)
    df_raw['log_income']          = np.log1p(df_raw['person_income'])
    df_raw['high_risk_intent']    = df_raw['loan_intent'].isin(
        ['DEBTCONSOLIDATION', 'MEDICAL']).astype(int)
    df_raw.to_parquet(PARQUET_IN, index=False)
    print(f"  → Reconstruit : {df_raw.shape}")

# Chargement
df = pd.read_parquet(PARQUET_IN)
print(f"Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Colonnes : {list(df.columns)}")

# ── Assertions obligatoires ──────────────────────────────────────────
assert df.shape == (32581, 15), (
    f"Shape inattendu : {df.shape} (attendu (32581, 15))"
)
assert 'loan_grade' not in df.columns, (
    "ERREUR : loan_grade présente — variable pré-octroi, ne doit jamais figurer dans le pipeline"
)
assert 'debt_service_rate' in df.columns, (
    "ERREUR : debt_service_rate absente — feature J1S2 manquante"
)

print("\n✓ Assertions passées : shape (32581, 15) | loan_grade absente | debt_service_rate présente")

# Taux de défaut global de référence
taux_defaut = df['loan_status'].mean() * 100
print(f"\n  Taux de défaut global : {taux_defaut:.1f} %")
print(f"  (Référence BCEAO : 21,8 % — benchmark de surrisque)")

df.head(3)

---
## Bloc 2 — Audit qualité : outliers & valeurs manquantes

> **Contexte métier :** Avant de nettoyer, on audite le dataset pour quantifier précisément les anomalies.  
> Un emprunteur de 144 ans ou avec 123 ans d'ancienneté professionnelle sont des erreurs de saisie courantes  
> dans les systèmes bancaires ivoiriens — le capping P99 est la réponse technique standard.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 2 — Audit qualité
# ═══════════════════════════════════════════════════════════════════

print("═" * 65)
print("AUDIT 1 — Valeurs manquantes")
print("═" * 65)
nan_summary = pd.DataFrame({
    'NaN count': df.isnull().sum(),
    'NaN %':    (df.isnull().sum() / len(df) * 100).round(1)
})
nan_summary = nan_summary[nan_summary['NaN count'] > 0]
print(nan_summary.to_string())
print(f"\n  loan_int_rate  : 3 116 NaN (9,6 %) → médiane = 10,99 %")
print(f"  person_emp_length : 895 NaN (2,7 %) → médiane calculée sur train")

print("\n" + "═" * 65)
print("AUDIT 2 — Outliers extrêmes (max et P99)")
print("═" * 65)
cols_outlier = ['person_age', 'person_emp_length', 'person_income']
for col in cols_outlier:
    max_val = df[col].max()
    p99_val = np.percentile(df[col].dropna(), 99)
    n_above = (df[col] > p99_val).sum()
    print(f"  {col:<25} max={max_val:>12,.0f}  P99={p99_val:>10,.0f}  ({n_above} obs au-dessus)")

print("\n  Valeurs anormales confirmées :")
print("    person_age max = 144 ans       → P99 = 50 ans (seuil de capping)")
print("    person_emp_length max = 123 ans → P99 = 18 ans (seuil de capping)")
print("    person_income max = 6 000 000 $ → P99 = 225 200 $")

In [ ]:
# Visualisation outliers — distribution person_age avant capping
# np.log1p() sur la colonne avant Plotly (log_x=True interdit dans px.histogram)
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df['person_age'].clip(upper=80),  # limiter l'axe pour lisibilité
    nbinsx=40,
    marker_color=TEAL,
    opacity=0.85,
    name='Distribution'
))
fig.add_vline(x=50, line_dash='dash', line_color=ORANGE,
              annotation_text='P99 = 50 ans (seuil capping)',
              annotation_position='top right',
              annotation_font_color=ORANGE)
fig.update_layout(
    title='Distribution person_age — Outliers visibles au-delà de P99',
    xaxis_title='Âge (années)',
    yaxis_title='Nombre d\'emprunteurs',
    paper_bgcolor=NAVY,
    plot_bgcolor='#0A2538',
    font_color='#C8DDE8',
    title_font_color=MINT
)
fig.show()

---
## Bloc 3 — Capping P99 (traitement des outliers)

> **Principe anti-leakage BCEAO :** Les seuils P99 seront réestimés sur le jeu d'entraînement dans le pipeline  
> Scikit-learn (jamais calculés sur l'ensemble complet avant `cross_val_score`).  
> Ici, on applique le capping sur le DataFrame pour produire le parquet de référence,  
> avec les valeurs P99 documentées comme constantes métier.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 3 — Capping P99
# RÈGLE ANTI-LEAKAGE : dans le pipeline Scikit-learn, P99 est recalculé
# uniquement sur le train set à chaque fold de cross-validation
# ═══════════════════════════════════════════════════════════════════

# Seuils P99 documentés (valeurs constantes métier pour le parquet de référence)
P99_AGE    = 50      # ans
P99_EMP    = 18      # ans
P99_INCOME = 225_200 # USD

print("Application du capping P99...")
print(f"  person_age        : max avant = {df['person_age'].max():.0f}  → plafond = {P99_AGE}")
print(f"  person_emp_length : max avant = {df['person_emp_length'].max():.0f}  → plafond = {P99_EMP}")
print(f"  person_income     : max avant = {df['person_income'].max():,.0f}  → plafond = {P99_INCOME:,}")

df_clean = df.copy()
df_clean['person_age']        = np.clip(df_clean['person_age'],        a_min=None, a_max=P99_AGE)
df_clean['person_emp_length'] = np.clip(df_clean['person_emp_length'],  a_min=None, a_max=P99_EMP)
df_clean['person_income']     = np.clip(df_clean['person_income'],      a_min=None, a_max=P99_INCOME)

# Vérification max après capping
print("\nVérification max après capping :")
print(f"  person_age        : {df_clean['person_age'].max():.0f}  (attendu ≤ {P99_AGE})")
print(f"  person_emp_length : {df_clean['person_emp_length'].max():.0f}  (attendu ≤ {P99_EMP})")
print(f"  person_income     : {df_clean['person_income'].max():,.0f}  (attendu ≤ {P99_INCOME:,})")

assert df_clean['person_age'].max()        <= P99_AGE,    f"Capping age échoué : {df_clean['person_age'].max()}"
assert df_clean['person_emp_length'].max() <= P99_EMP,    f"Capping emp échoué : {df_clean['person_emp_length'].max()}"
assert df_clean['person_income'].max()     <= P99_INCOME, f"Capping income échoué : {df_clean['person_income'].max()}"

print("\n✓ Capping P99 appliqué et validé")

# Impact sur la distribution person_income
n_capped = (df['person_income'] > P99_INCOME).sum()
print(f"  Observations cappées sur person_income : {n_capped} ({n_capped/len(df)*100:.1f} %)")

---
## Bloc 4 — Imputation médiane des valeurs manquantes

> **Choix de l'imputation médiane vs moyenne :**  
> La médiane est robuste aux outliers — après capping P99, les distributions restent asymétriques.  
> La médiane de `loan_int_rate` = **10,99 %** est calculée sur l'ensemble et utilisée ici.  
> Dans le pipeline Scikit-learn Jour 2, `SimpleImputer(strategy='median')` recalcule la médiane  
> **uniquement sur le train set** à chaque fold pour éviter le data leakage.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 4 — Imputation médiane
# ═══════════════════════════════════════════════════════════════════

print("NaN avant imputation :")
cols_nan = ['loan_int_rate', 'person_emp_length']
for col in cols_nan:
    n = df_clean[col].isnull().sum()
    m = df_clean[col].median()
    print(f"  {col:<25} : {n:>5} NaN  →  médiane = {m:.2f}")

# Imputation par médiane
for col in cols_nan:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

# Vérification
print("\nNaN après imputation :")
for col in cols_nan:
    n = df_clean[col].isnull().sum()
    print(f"  {col:<25} : {n} NaN  ✓")

# loan_int_rate médiane de référence
median_int_rate = df_clean['loan_int_rate'].median()
print(f"\n  loan_int_rate médiane après imputation : {median_int_rate:.2f} %")
print(f"  (Référence attendue : 10,99 %)")

total_nan = df_clean.isnull().sum().sum()
print(f"\n  Total NaN restants dans le dataset : {total_nan}")
if total_nan == 0:
    print("✓ Dataset sans NaN après imputation")
else:
    print("⚠ NaN résiduels à traiter :", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

---
## Bloc 5 — Encodage catégoriel

> **Contexte métier :** Pour les algorithmes ML (RF, GB, LR), toutes les variables doivent être numériques.  
> - `person_home_ownership` → dummies (home_RENT, home_MORTGAGE, home_OWN) — drop_first supprime OTHER  
> - `cb_person_default_on_file` → binaire : Y=1, N=0 (historique de paiement — disponible en banque ivoirienne)  
> - `loan_intent` → LabelEncoder (capturé partiellement par `high_risk_intent` déjà présente)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 5 — Encodage catégoriel
# ═══════════════════════════════════════════════════════════════════

# 5.1 — person_home_ownership → dummies (drop_first=True supprime OTHER)
home_dummies = pd.get_dummies(
    df_clean['person_home_ownership'],
    prefix='home',
    drop_first=True  # supprime home_MORTGAGE (référence → OTHER devient implicite)
)
# On veut explicitement : home_RENT, home_MORTGAGE, home_OWN
# Approche : garder toutes les dummies sauf la catégorie de référence (OTHER = freq la plus basse)
home_dummies_full = pd.get_dummies(
    df_clean['person_home_ownership'],
    prefix='home'
)
# Taux de défaut : RENT=31,6% > OTHER=30,8% > MORTGAGE=12,6% > OWN=7,5%
# On garde : home_RENT, home_MORTGAGE, home_OWN (drop OTHER comme catégorie moins discriminante)
home_cols = ['home_RENT', 'home_MORTGAGE', 'home_OWN']
for col in home_cols:
    if col not in home_dummies_full.columns:
        home_dummies_full[col] = 0
df_clean = pd.concat([df_clean, home_dummies_full[home_cols]], axis=1)
print(f"✓ Dummies home_ownership créées : {home_cols}")

# 5.2 — cb_person_default_on_file → binaire Y=1, N=0
df_clean['default_enc'] = df_clean['cb_person_default_on_file'].map({'Y': 1, 'N': 0})
# Vérification
n_nan_default = df_clean['default_enc'].isnull().sum()
print(f"✓ default_enc créée (Y=1, N=0) — {n_nan_default} NaN")
print(f"  Distribution : {df_clean['default_enc'].value_counts().to_dict()}")
print(f"  Taux de défaut Y={df_clean[df_clean['default_enc']==1]['loan_status'].mean()*100:.1f}% | N={df_clean[df_clean['default_enc']==0]['loan_status'].mean()*100:.1f}%")

# 5.3 — loan_intent → LabelEncoder
le_intent = LabelEncoder()
df_clean['loan_intent_enc'] = le_intent.fit_transform(df_clean['loan_intent'])
print(f"\n✓ loan_intent encodé : {dict(zip(le_intent.classes_, le_intent.transform(le_intent.classes_)))}")

# Colonnes catégorielles sources — drop (remplacées par leur encodage)
# Note : on conserve les colonnes sources pour traçabilité
print(f"\nColonnes après encodage : {df_clean.shape[1]}")
print(f"Nouvelles colonnes : default_enc, loan_intent_enc, home_RENT, home_MORTGAGE, home_OWN")

---
## Bloc 6 — Validation finale

> **Critères pour un dataset ML-ready (K-Means Jour 2) :**  
> 1. Shape ~(32 581, 21–24) selon dummies créées  
> 2. **0 NaN** — K-Means ne tolère pas les valeurs manquantes  
> 3. Toutes colonnes numériques ou booléennes — pas de string  
> 4. `debt_service_rate` présente (feature signal #1 Spearman=+0,389)  
> 5. `loan_grade` absente (vérification finale anti-leakage)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 6 — Validation finale
# ═══════════════════════════════════════════════════════════════════

print("═" * 60)
print("VALIDATION FINALE — credit_risk_clean.parquet")
print("═" * 60)

# 1. Shape
n_rows, n_cols = df_clean.shape
print(f"\n1. Shape : {n_rows:,} × {n_cols}")
assert n_rows == 32581, f"Nombre de lignes incorrect : {n_rows}"
assert 20 <= n_cols <= 26, f"Nombre de colonnes hors plage attendue : {n_cols}"
print(f"   ✓ {n_rows:,} lignes × {n_cols} colonnes (attendu : 32 581 × ~20–24)")

# 2. Zéro NaN
total_nan = df_clean.isnull().sum().sum()
print(f"\n2. NaN total : {total_nan}")
if total_nan == 0:
    print("   ✓ Dataset sans NaN — prêt pour K-Means")
else:
    cols_with_nan = df_clean.isnull().sum()[df_clean.isnull().sum() > 0]
    print(f"   ⚠ NaN résiduels dans : {dict(cols_with_nan)}")
    # Imputation de secours
    df_clean = df_clean.fillna(df_clean.median(numeric_only=True))
    print("   → Imputation de secours appliquée")

# 3. Types
object_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"\n3. Colonnes de type object : {object_cols}")
print("   (Les colonnes sources textuelles sont conservées pour traçabilité)")

# 4. Features clés
features_requises = [
    'debt_service_rate', 'monthly_payment_proxy', 'log_income', 'high_risk_intent',
    'home_RENT', 'home_MORTGAGE', 'home_OWN', 'default_enc', 'loan_intent_enc'
]
print("\n4. Features requises :")
for feat in features_requises:
    status = "✓" if feat in df_clean.columns else "✗ MANQUANT"
    print(f"   {status} {feat}")

# 5. Absence loan_grade (anti-leakage)
assert 'loan_grade' not in df_clean.columns, "ERREUR CRITIQUE : loan_grade présente dans le parquet final !"
print("\n5. ✓ loan_grade absente — anti-leakage validé")

# 6. Taux de défaut conservé
taux = df_clean['loan_status'].mean() * 100
assert abs(taux - 21.8) < 0.5, f"Taux de défaut anormal : {taux:.1f}%"
print(f"\n6. ✓ Taux de défaut : {taux:.1f}% (référence : 21,8%)")

print("\n" + "═" * 60)
print("✓ VALIDATION RÉUSSIE — Prêt pour sauvegarde Parquet")
print("═" * 60)

# Aperçu des colonnes finales
print(f"\nColonnes finales :")
for i, col in enumerate(df_clean.columns):
    dtype = df_clean[col].dtype
    print(f"  [{i:02d}] {col:<30} {str(dtype)}")

---
## Bloc 7 — Sauvegarde credit_risk_clean.parquet

> **Format Parquet vs CSV :**  
> Parquet compresse les données (~3× plus petit que CSV), préserve les types,  
> et est 10× plus rapide à lire en Pandas. Standard dans tous les pipelines bancaires modernes.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 7 — Sauvegarde Parquet
# Chemin : ROOT / 'data' / 'processed' / 'credit_risk_clean.parquet'
# ═══════════════════════════════════════════════════════════════════

df_clean.to_parquet(PARQUET_OUT, index=False)
print(f"✓ Parquet sauvegardé : {PARQUET_OUT}")

# Vérification relecture
df_check = pd.read_parquet(PARQUET_OUT)
print(f"\nVérification relecture :")
print(f"  Shape   : {df_check.shape}")
print(f"  NaN     : {df_check.isnull().sum().sum()}")
print(f"  Défaut  : {df_check['loan_status'].mean()*100:.1f} %")

# Taille fichier
size_mb = PARQUET_OUT.stat().st_size / 1024 / 1024
print(f"  Taille  : {size_mb:.2f} Mo")

assert df_check.shape == df_clean.shape, "Erreur relecture — shape différent"
assert df_check.isnull().sum().sum() == 0, "Erreur relecture — NaN présents"

print("\n✓ credit_risk_clean.parquet validé — prêt pour J2S1 K-Means")
print("\n  Prochain fichier dans la chaîne Parquet :")
print(f"  {DATA_PROCESSED / 'credit_risk_kmeans.parquet'}")
print(f"  (J2S1 — K-Means Risk Profiling, 4 profils de risque)")

---
## Bloc 8 — Visualisation récapitulative : importance Spearman des features

> Signal comparé entre features originales et features engineered (J1S2 + J1S4).


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 8 — Visualisation récapitulative Spearman
# ═══════════════════════════════════════════════════════════════════

from scipy import stats

# Features numériques disponibles
num_features = [
    'debt_service_rate',
    'loan_percent_income',
    'loan_int_rate',
    'monthly_payment_proxy',
    'person_income',
    'default_enc',
    'home_RENT',
    'high_risk_intent',
    'loan_amnt',
    'person_emp_length',
    'person_age',
    'log_income',
    'cb_person_cred_hist_length'
]

corrs = []
for feat in num_features:
    if feat in df_clean.columns:
        rho, _ = stats.spearmanr(
            df_clean[feat].fillna(0),
            df_clean['loan_status']
        )
        corrs.append({'feature': feat, 'spearman': round(rho, 3)})

df_corr = pd.DataFrame(corrs).sort_values('spearman', key=abs, ascending=True)

colors = [MINT if r > 0 else ORANGE for r in df_corr['spearman']]
fig = go.Figure(go.Bar(
    x=df_corr['spearman'],
    y=df_corr['feature'],
    orientation='h',
    marker_color=colors,
    text=[f"{v:+.3f}" for v in df_corr['spearman']],
    textposition='outside'
))
fig.add_vline(x=0, line_color='white', line_width=1)
fig.update_layout(
    title='Corrélation Spearman vs loan_status (défaut) — J1S4 recalculé',
    xaxis_title='Spearman r',
    paper_bgcolor=NAVY,
    plot_bgcolor='#0A2538',
    font_color='#C8DDE8',
    title_font_color=MINT,
    height=500
)
fig.show()
print("\nTop 3 signaux :")
df_top = df_corr.reindex(df_corr['spearman'].abs().sort_values(ascending=False).index).head(3)
for _, row in df_top.iterrows():
    print(f"  {row['feature']:<30} Spearman = {row['spearman']:+.3f}")

---
### 📊 Interprétation du graphique Spearman

> **Règle de lecture :** barre verte vers la droite → la variable augmente avec le risque de défaut.  
> Barre orange vers la gauche → la variable protège contre le défaut.  
> La longueur = la force du signal (valeur absolue du ρ).

---

#### 🔴 Groupe 1 — Signaux forts (|ρ| > 0,25) : les moteurs du risque

| Feature | ρ | Lecture métier |
|---------|---|----------------|
| `debt_service_rate` | **+0,389** | Feature engineered = taux × charge/revenu. Capture la **double pression financière** — un emprunteur qui paie cher ET consacre une grande part de son revenu. Signal #1 du dataset. |
| `monthly_payment_proxy` | **+0,322** | Mensualité estimée / revenu mensuel. Bat `loan_percent_income` (+0,316) car il ramène la charge à l'échelle de la **trésorerie mensuelle** — ce que vit réellement le ménage. |
| `loan_percent_income` | **+0,316** | Charge du prêt rapportée au revenu annuel. Variable source, mais légèrement moins puissante que ses dérivées. |
| `loan_int_rate` | **+0,298** | Taux élevé = risque perçu élevé par le prêteur. **Réhabilité** : avec `loan_grade` (r=0,89), son signal était absorbé — sans le grade, il révèle sa valeur propre. |
| `log_income` / `person_income` | **−0,272** | Même ρ car `log1p()` préserve le rang des individus (Spearman est basé sur les rangs). Revenu élevé = **coussin financier** qui absorbe les chocs. |

**Ce que ce groupe dit :** le risque de défaut est avant tout **financier et structurel** — ratio charge/revenu, coût du crédit. Ce n'est pas un risque de profil démographique.

---

#### 🟡 Groupe 2 — Signaux modérés (0,10 < |ρ| < 0,25) : discriminants réels mais partiels

| Feature | ρ | Lecture métier |
|---------|---|----------------|
| `home_RENT` | **+0,238** | Être locataire → charge fixe supplémentaire qui réduit la marge de manœuvre. En contexte ivoirien : aussi un proxy de stabilité patrimoniale (un propriétaire a un actif implicite). |
| `default_enc` | **+0,179** | Défaut passé prédit un nouveau défaut — mais à 0,179 seulement, ce n'est pas déterministe. L'état financier **actuel** (revenus, charges) compte davantage que l'historique. |
| `high_risk_intent` | **+0,101** | DEBTCONSOLIDATION ou MEDICAL : quelqu'un qui consolide ses dettes existantes est déjà en tension. Signal orthogonal aux ratios financiers (r≈0,002) — apport marginal réel. |

---

#### ⚪ Groupe 3 — Signaux faibles (|ρ| < 0,10) : peu d'information directe

| Feature | ρ | Lecture métier |
|---------|---|----------------|
| `person_emp_length` | −0,096 | L'ancienneté professionnelle protège légèrement, mais si les charges sont disproportionnées, un CDI de 10 ans ne suffit pas. |
| `loan_amnt` | +0,084 | Le montant seul ne dit presque rien — c'est son rapport au revenu qui compte (`loan_percent_income`). |
| `person_age` | −0,033 | L'âge ne prédit pratiquement pas le défaut. Signal quasi-nul. |
| `cb_person_cred_hist_length` | −0,024 | La **durée** d'historique de crédit ≠ la **qualité**. Un long historique peut inclure des défauts — ce signal est trop ambigu pour être utile. |

---

#### ✅ 3 enseignements de fond

**1. Le Feature Engineering a fonctionné.**  
`debt_service_rate` (FE, +0,389) > `monthly_payment_proxy` (FE, +0,322) > `loan_percent_income` (source, +0,316) > `loan_int_rate` (source, +0,298).  
Les deux features construites **battent leurs composantes séparées** — l'interaction capture quelque chose que les variables brutes ne portent pas individuellement.

**2. `log_income` et `person_income` ont le même ρ — et c'est normal.**  
Spearman est basé sur les **rangs**, pas les valeurs. `log1p()` est une transformation monotone croissante : elle ne change pas l'ordre des individus, donc pas les rangs, donc pas le ρ.  
La différence entre les deux apparaîtra dans les modèles linéaires (Logistic Regression) où `log_income` stabilise les coefficients face aux queues de distribution.

**3. Le risque est situationnel, pas démographique.**  
Age, ancienneté, historique de crédit → tous faibles. Les 4 premiers signaux sont des ratios financiers.  
Ce dataset modélise une population dont le risque dépend de **ce qu'elle doit rembourser par rapport à ce qu'elle gagne** — pas de qui elle est.

> **Pour le Jour 2 :** ces ρ guident la sélection de features pour le K-Means (J2S1) et confirment  
> que `debt_service_rate` et `monthly_payment_proxy` doivent être au cœur du pipeline RF (J2S2).


---
## Bloc 9 — Commit Git

> **Bonne pratique :** chaque session se termine par un commit traçable. Exécuter ces commandes dans le **terminal** (VS Code) ou dans une **cellule Colab** préfixée par `!`.

### Commit Git

```bash
git add data/processed/credit_risk_clean.parquet
git add notebooks/j1s4_nettoyage_fe.ipynb
git commit -m "feat(j1s4): credit_risk_clean.parquet ML-ready — capping P99, imputation médiane, encodage"
git push origin main
```

Vérification :
```bash
git log --oneline -3
```

Résultat attendu :
```
abc1234 feat(j1s4): credit_risk_clean.parquet ML-ready — capping P99, imputation médiane, encodage
def5678 feat(j1s3): EDA Plotly — 9 figures sans loan_grade, debt_service_rate
ghi9012 feat(j1s2): credit_features_j1.parquet — pipeline sans loan_grade
```

> **Sur Colab :** préfixer chaque commande par `!` — ex. `!git add ...`


---
## Récapitulatif J1S4 — Ce que vous avez produit

| Étape | Action | Résultat |
|-------|--------|----------|
| **Capping P99** | Clip outliers (144→50 ans, 123→18 ans) | Distributions normalisées |
| **Imputation** | Médiane loan_int_rate (10,99%) + emp_length | 0 NaN dans le dataset |
| **Dummies** | home_RENT, home_MORTGAGE, home_OWN | Propriété du logement encodée |
| **Binaire** | default_enc Y=1/N=0 | Historique de défaut encodé |
| **Label** | loan_intent_enc | Intention de prêt encodée |
| **Validation** | 0 NaN · loan_grade absente · 21,8% défaut | Dataset ML-ready |
| **Sauvegarde** | credit_risk_clean.parquet | Livrable J1S4 |

### Features signal disponibles pour le Jour 2

| Feature | Spearman | Importance GB | Source |
|---------|----------|---------------|--------|
| `debt_service_rate` | **+0,389** | 8,9 % | FE J1S2 |
| `loan_percent_income` | +0,316 | — | Source |
| `loan_int_rate` | +0,298 | **20,4 %** | Réhabilité (sans grade) |
| `monthly_payment_proxy` | +0,322 | **28,6 %** | FE J1S2 |
| `person_income` | −0,272 | 8,7 % | Source |

> **Demain J2S1 :** K-Means sur ces features → 4 profils de risque client  
> **Demain J2S2 :** Random Forest Baseline → AUC = 0,929
